In [ ]:
# Supressing cell output from spotpy
from datetime import date, timedelta
from contextlib import contextmanager
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import pdm_model26
#import BaseflowSeparation
import proplot as pplt
from scipy.stats import spearmanr
import spotpy
#from sklearn.metrics import r2_score
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings("ignore")


plot

In [ ]:
def kgeprime(evaluation, simulation):
    """Modified Kling-Gupta Efficiency (KGE') and its three components
    (r, γ, β) as per `Kling et al., 2012
    """
    # calculate error in timing and dynamics r
    # (Pearon's correlation coefficient)
    sim_mean = np.mean(simulation, axis=0, dtype=np.float64)
    obs_mean = np.mean(evaluation, dtype=np.float64)

    r_num = np.sum((simulation - sim_mean) * (evaluation - obs_mean),
                axis=0, dtype=np.float64)
    r_den = np.sqrt(np.sum((simulation - sim_mean) ** 2,
                        axis=0, dtype=np.float64)
                    * np.sum((evaluation - obs_mean) ** 2,
                            dtype=np.float64))
    r = r_num / r_den

    # calculate error in spread of flow gamma
    # (avoiding cross correlation with bias by dividing by the mean)
    gamma = ((np.std(simulation, axis=0, dtype=np.float64) / sim_mean)
            / (np.std(evaluation, dtype=np.float64) / obs_mean))

    # calculate error in volume beta (bias of mean discharge)
    beta = (np.mean(simulation, axis=0, dtype=np.float64)
            / np.mean(evaluation, axis=0, dtype=np.float64))

    # calculate the modified Kling-Gupta Efficiency KGE'
    kgeprime_ = 1 - np.sqrt((r - 1) ** 2 + (gamma - 1) ** 2 + (beta - 1) ** 2)

    return kgeprime_

In [ ]:
kges = pd.DataFrame(columns=['kge','AI', 'stationid'])
dir_topo = 'D:/yuanqi/new urania partition/camels_600.txt'
camels_info = pd.read_csv(dir_topo, delimiter=';')
camels_info = camels_info.sort_values(by='aridity').reset_index(drop=True)
gauge_id = camels_info['gauge_id'].values.astype(np.int64)
for ii in range(len(gauge_id)):
    stationid = gauge_id[ii]
    stationid = float(stationid)
    camels_info['gauge_id'] = camels_info['gauge_id'].astype(float)
    aridity = camels_info[camels_info['gauge_id'] == stationid]['aridity'].iloc[0]
    aridity = round(aridity,3)
    stationid = "%08d"%stationid
    # kge
    outd_file = pd.read_csv('D:/yuanqi/1106/sim/'+ str(stationid) + '_AI_' + str(aridity) + '_simulated26.csv',index_col=0)
    outd_calib = outd_file[730:7670]
    outd_valid = outd_file[7670:]


    
    simq_calib = outd_calib['Qsim']
    obsq_calib = outd_calib['Qobs']

    simq_valid = outd_valid['Qsim']
    obsq_valid = outd_valid['Qobs']

    kge_calib = kgeprime(np.array(obsq_calib),np.array(simq_calib))
    kge_valid = kgeprime(np.array(obsq_valid),np.array(simq_valid))


    #print(kgeold)
    kges.at[ii,'kge_calib'] = kge_calib
    kges.at[ii,'kge_valid'] = kge_valid
    kges.at[ii,'Aridity Index'] = aridity
    kges.at[ii,'stationid'] = stationid

kges.to_csv('D:/yuanqi/new urania partition/kges_cali_vali.csv') 

In [ ]:
import pandas as pd
import numpy as np
import proplot as pplt

# Load data
kges = pd.read_csv("D:/yuanqi/new urania partition/kges_cali_vali.csv")
kges["climate"] = ["Arid" if ai > 1 else "Humid" for ai in kges["Aridity Index"]]

# Prepare numeric arrays
def clean(s):
    arr = pd.to_numeric(s, errors='coerce')
    return arr.dropna().to_numpy()

arid_calib = clean(kges.loc[kges["climate"]=="Arid", "kge_calib"])
arid_valid = clean(kges.loc[kges["climate"]=="Arid", "kge_valid"])
humid_calib = clean(kges.loc[kges["climate"]=="Humid", "kge_calib"])
humid_valid = clean(kges.loc[kges["climate"]=="Humid", "kge_valid"])

data_list = [arid_calib, arid_valid, humid_calib, humid_valid]

# Pad arrays to same length
max_len = max(len(d) for d in data_list)
data_array = np.column_stack([np.pad(d, (0, max_len-len(d)), constant_values=np.nan) for d in data_list])

labels = ["Calib–Arid", "Valid–Arid", "Calib–Humid", "Valid–Humid"]

fig, ax = pplt.subplots(figsize=(6,6))  # smaller figure

colors = ['skyblue', 'lightgreen', 'skyblue', 'lightgreen']  # Calib, Valid pattern

bp = ax.boxplot(
    data_array,
    labels=labels,
    widths=0.45,
    fillcolors=colors,  # assign colors per box
    linewidth=2,
    medianprops=dict(color='red', linewidth=2.5)
)

# Adjust font sizes for smaller figure
ax.set_ylabel("KGE", fontsize=14)
#ax.set_title("KGE Performance by Climate (Calib vs Valid)", fontsize=16)
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)
fig.savefig("D:/yuanqi/new urania partition/kge_boxplot.png", dpi=300, bbox_inches='tight')
pplt.show()


